# Robust Classification Under Noisy Labels — Flowers Dataset

**CS 484 Final Project**

This notebook is the single entry point for the entire project. Run cells top-to-bottom to reproduce all results and figures.

---

## Pipeline overview

| Phase | What happens |
|---|---|
| **1 — Feature extraction** | Load flower images → DINO ViT-S/16 → cache 384-dim embeddings |
| **2 — Noise infrastructure** | Apply uniform / asymmetric / instance-dependent / human noise |
| **3 — Methods** | 8 noise-robust classifiers with a consistent `fit` / `predict` interface |
| **4A — Synthetic sweep** | Grid over 3 noise types × 6 noise rates × 3 seeds × 8 methods |
| **4B — Real data** | Evaluate on human annotations from the Flask annotation tool |
| **5 — Figures** | 7 publication-ready plots from the saved JSON results |

**Dataset:** 4 317 Flowers images, 5 classes (daisy · dandelion · rose · sunflower · tulip), fixed 15 % validation split that is *never* corrupted.

---
## Setup & imports

In [ ]:
import sys, json, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
warnings.filterwarnings('ignore')

# ROOT  = directory containing notebook.ipynb  (Noisy-Label-Classifier/)
# MYLIBS = mylibs/ subfolder where all code and data live
ROOT   = Path().resolve()
MYLIBS = ROOT / 'mylibs'

# Add mylibs/ to path so noise/, methods/, experiments/, visualize/, dataset.py
# are all importable without any prefix.
if str(MYLIBS) not in sys.path:
    sys.path.insert(0, str(MYLIBS))

print(f"Project root : {ROOT}")
print(f"mylibs/      : {MYLIBS}")
print(f"Python       : {sys.version.split()[0]}")

import torch
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}  "
      f"({'GPU: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

---
## Phase 1 — Feature Extraction

Run **once**. Downloads DINO from torch hub and extracts 384-dim embeddings for all 4 317 flower images.  
Saves `features.npy`, `labels.npy`, `image_paths.npy`, `val_indices.npy` to `mylibs/`.

Skip this cell if those files already exist.

In [ ]:
FEATURES_FILE = MYLIBS / 'features.npy'

if FEATURES_FILE.exists():
    print("features.npy already exists — skipping extraction.")
    print("Delete mylibs/features.npy and re-run this cell to re-extract.")
else:
    print("Extracting DINO features (this takes a few minutes) …")
    from extract_features import extract_dino_features
    extract_dino_features()

In [ ]:
# Load cached data — used by every subsequent cell
from dataset import load_data, CLASSES

features, labels, image_paths, train_idx, val_idx = load_data(MYLIBS)

train_features = features[train_idx]
train_labels   = labels[train_idx]
val_features   = features[val_idx]
val_labels     = labels[val_idx]

print(f"Total samples  : {len(labels)}")
print(f"Train / Val    : {len(train_idx)} / {len(val_idx)}")
print(f"Feature dim    : {features.shape[1]}")
print(f"Classes        : {CLASSES}")
print()

# Class distribution
for i, cls in enumerate(CLASSES):
    n_tr = (train_labels == i).sum()
    n_va = (val_labels   == i).sum()
    print(f"  {cls:12s}  train={n_tr:4d}  val={n_va:3d}")

---
## Phase 2 — Noise Infrastructure

Four noise types available via the `apply_noise()` factory:

| Type | Description |
|---|---|
| `uniform` | Every label flips to a random wrong class with probability *p* |
| `asymmetric` | Labels only flip within confusable pairs (rose↔tulip, daisy↔sunflower, dandelion→daisy) |
| `instance` | Samples far from their class centroid are mislabeled at higher rates |
| `human` | Labels replaced by human annotations from the Flask annotation tool |

In [ ]:
from noise.factory import apply_noise

NOISE_RATE = 0.3

demos = {
    'uniform':    apply_noise(train_features, train_labels, 'uniform',    NOISE_RATE),
    'asymmetric': apply_noise(train_features, train_labels, 'asymmetric', NOISE_RATE),
    'instance':   apply_noise(train_features, train_labels, 'instance',   NOISE_RATE),
}

print(f"Noise demonstrations at requested rate = {NOISE_RATE}\n")
print(f"{'Type':16s}  {'Actual rate':>12s}  {'Flipped':>8s}")
print("-" * 42)
for name, noisy in demos.items():
    actual = (noisy != train_labels).mean()
    n_flip = (noisy != train_labels).sum()
    print(f"{name:16s}  {actual:12.3f}  {n_flip:8d}")

In [ ]:
# Visualise which class pairs are affected by each noise type
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Noise Pattern at {int(NOISE_RATE*100)}% — per-class flip destination', fontsize=12)

for ax, (name, noisy) in zip(axes, demos.items()):
    matrix = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    for t, n in zip(train_labels, noisy):
        matrix[t, n] += 1

    im = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=8)
    ax.set_xlabel('Assigned label'); ax.set_ylabel('True label')
    ax.set_title(name.capitalize())
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center',
                    fontsize=7, color='white' if matrix[i,j] > matrix.max()*0.5 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

### Human annotation noise
Run the Flask annotation tool first (`python run_pipeline.py --phase annotate`), then execute the cell below to see the effective noise rate from the collected labels.

In [ ]:
import csv
from collections import Counter

ANNOTATIONS_CSV = MYLIBS / 'manual_annotations.csv'

if not ANNOTATIONS_CSV.exists() or ANNOTATIONS_CSV.stat().st_size < 50:
    print("manual_annotations.csv is empty — run the annotation tool first.")
    print("  python mylibs/run_pipeline.py --phase annotate")
else:
    human_noisy = apply_noise(
        train_features, labels, 'human',
        image_paths=image_paths,
        train_idx=train_idx,
        annotations_csv=str(ANNOTATIONS_CSV),
    )
    actual_rate = (human_noisy != train_labels).mean()
    n_annotated = sum(1 for r in csv.DictReader(open(ANNOTATIONS_CSV))
                      if int(r['label']) != -1)

    print(f"Annotations collected : {n_annotated}")
    print(f"Effective noise rate  : {actual_rate:.3f}")

    # Per-class breakdown
    print("\nPer-class error rate (annotated samples only):")
    for i, cls in enumerate(CLASSES):
        mask = train_labels == i
        n    = mask.sum()
        if n == 0: continue
        wrong = (human_noisy[mask] != train_labels[mask]).sum()
        print(f"  {cls:12s}  {wrong}/{n}  ({wrong/n:.1%})")

---
## Phase 3 — Methods

Six methods all share `fit(features, noisy_labels)` / `predict(features)`:

| Method | Core idea |
|---|---|
| `BaselineCE` | Standard cross-entropy — lower bound |
| `LabelSmoothing` | Spread ε probability mass to prevent overconfidence |
| `SCE` | Symmetric CE: add a reverse-CE term for robustness |
| `GCE` | Generalised CE: q-loss interpolates between CE (q→1) and MAE (q→0) |
| `GMMReweight` | Fit 2-component GMM to losses; upweight clean-component samples |
| `ConfidentLearning` | OOF probs → flag likely mislabelled → prune → retrain |

The cell below runs a **quick single-condition sanity check** (30% uniform noise) so you can verify all methods work before launching the full sweep.

In [ ]:
from methods import (BaselineCE, LabelSmoothing, SCE, GCE,
                     GMMReweight, ConfidentLearning)

# Quick sanity-check: 30% uniform noise, 40 epochs
QUICK_EPOCHS = 40
QUICK_NOISE  = 0.3

noisy_quick = apply_noise(train_features, train_labels, 'uniform', QUICK_NOISE)

kw = dict(val_features=val_features, val_labels=val_labels, epochs=QUICK_EPOCHS)
sanity_methods = [
    BaselineCE(**kw),
    LabelSmoothing(**kw),
    SCE(**kw),
    GCE(**kw),
    GMMReweight(warmup_epochs=12, total_epochs=QUICK_EPOCHS, **kw),
    ConfidentLearning(warmup_epochs=QUICK_EPOCHS // 2, total_epochs=QUICK_EPOCHS, **kw),
]

print(f"Sanity check — uniform {int(QUICK_NOISE*100)}% noise, {QUICK_EPOCHS} epochs\n")
print(f"{'Method':20s}  {'Val Acc':>8s}")
print("-" * 32)
for m in sanity_methods:
    t0 = time.time()
    m.fit(train_features, noisy_quick)
    acc  = m.evaluate(val_features, val_labels)
    elapsed = time.time() - t0
    print(f"{m.name:20s}  {acc:8.4f}  ({elapsed:.1f}s)")

---
## Phase 4A — Synthetic Sweep

Full grid: **3 noise types × 6 noise rates × 3 seeds × 8 methods = 432 training runs**.

Results are saved to `results/synthetic_results.json` and `results/synthetic_aux.json`.  
**Re-running this cell will overwrite existing results.**

> ⏱ Estimated time: ~20 min on GPU, ~3–5 hours on CPU.

In [ ]:
from experiments.synthetic_sweep import run as run_synthetic_sweep

run_synthetic_sweep()

In [ ]:
# Quick summary table of synthetic results
SYNTHETIC_FILE = MYLIBS / 'results' / 'synthetic_results.json'

if not SYNTHETIC_FILE.exists():
    print("synthetic_results.json not found — run the sweep above first.")
else:
    with open(SYNTHETIC_FILE) as f:
        syn_results = json.load(f)

    NOISE_TYPES  = ['uniform', 'asymmetric', 'instance']
    NOISE_RATES  = [0.0, 0.05, 0.1, 0.2, 0.4, 0.6]
    METHOD_ORDER = ['BaselineCE', 'LabelSmoothing', 'SCE', 'GCE',
                    'GMMReweight', 'ConfidentLearning']

    for noise_type in NOISE_TYPES:
        print(f"\n{'='*70}")
        print(f"  Noise type: {noise_type}")
        print(f"{'='*70}")
        header = f"{'Method':20s}" + "".join(f"  nr={nr:.1f}" for nr in NOISE_RATES)
        print(header)
        print("-" * len(header))
        for m in METHOD_ORDER:
            row = f"{m:20s}"
            for nr in NOISE_RATES:
                accs = syn_results.get(noise_type, {}).get(str(float(nr)), {}).get(m, [])
                if accs:
                    row += f"  {np.mean(accs):.3f}"
                else:
                    row += "    — "
            print(row)

---
## Phase 4B — Real Data Experiment

Evaluates all methods on four label sets:

| Label set | Description |
|---|---|
| `clean` | Original ground-truth labels (upper bound) |
| `human` | Human annotations from the Flask annotation tool |
| `uniform_30` | 30% uniform noise (synthetic reference) |
| `asymmetric_30` | 30% asymmetric noise (synthetic reference) |

> Run the annotation tool before this phase to populate `manual_annotations.csv`.

In [ ]:
from experiments.real_data import run as run_real_experiment

run_real_experiment()

In [ ]:
# Summary table for real data results
REAL_FILE = MYLIBS / 'results' / 'real_results.json'

if not REAL_FILE.exists():
    print("real_results.json not found — run the experiment above first.")
else:
    with open(REAL_FILE) as f:
        real_data = json.load(f)

    results    = real_data['results']
    noise_info = real_data['noise_rates']
    label_sets = real_data['label_sets']

    print(f"{'Method':20s}", end='')
    for ls in label_sets:
        nr = noise_info.get(ls, 0)
        print(f"  {ls[:12]:>14s}", end='')
    print()
    print(f"{'':20s}", end='')
    for ls in label_sets:
        nr = noise_info.get(ls, 0)
        print(f"  {'(nr='+f'{nr:.2f}'+')':>14s}", end='')
    print()
    print("-" * (20 + 16 * len(label_sets)))

    METHOD_ORDER = ['BaselineCE', 'LabelSmoothing', 'SCE', 'GCE',
                    'GMMReweight', 'ConfidentLearning']
    for m in METHOD_ORDER:
        print(f"{m:20s}", end='')
        for ls in label_sets:
            acc = results.get(ls, {}).get(m, None)
            print(f"  {acc:14.4f}" if acc is not None else f"  {'—':>14s}", end='')
        print()

---
## Phase 5 — Figures

All plots read from the saved JSON files — no models are re-trained.

| Figure | Content |
|---|---|
| **Fig 1** | Accuracy vs noise rate — one subplot per noise type |
| **Fig 2** | GMM loss histogram — bimodal clean/noisy distribution |
| **Fig 3** | Transition matrix — CL estimate vs ground truth |
| **Fig 4** | Flagged sample grid — images CL identified as mislabelled |
| **Fig 5** | Noise rate calibration — CL estimated vs true rate |
| **Fig 6** | Learning curves — val accuracy over epochs at 40% uniform noise |
| **Fig 7** | Real data bar chart — human annotations vs synthetic baselines |

### Figure 1 — Accuracy vs. Noise Rate

In [ ]:
SYNTHETIC_FILE = MYLIBS / 'results' / 'synthetic_results.json'

if not SYNTHETIC_FILE.exists():
    print("Run Phase 4A first.")
else:
    with open(SYNTHETIC_FILE) as f:
        syn_results = json.load(f)

    NOISE_TYPES  = ['uniform', 'asymmetric', 'instance']
    NOISE_LABELS = ['Uniform', 'Asymmetric', 'Instance-Dependent']
    NOISE_RATES  = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    METHOD_ORDER = ['BaselineCE', 'LabelSmoothing', 'SCE', 'GCE',
                    'GMMReweight', 'ConfidentLearning']
    colors = cm.tab10(np.linspace(0, 0.8, len(METHOD_ORDER)))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    fig.suptitle('Figure 1 — Validation Accuracy vs. Noise Rate', fontsize=13)

    for ax, nt, nl in zip(axes, NOISE_TYPES, NOISE_LABELS):
        nt_data = syn_results.get(nt, {})
        for m_name, color in zip(METHOD_ORDER, colors):
            means, stds = [], []
            for nr in NOISE_RATES:
                accs = nt_data.get(str(float(nr)), {}).get(m_name, [])
                means.append(np.mean(accs) if accs else np.nan)
                stds.append(np.std(accs)  if accs else 0.0)
            means, stds = np.array(means), np.array(stds)
            valid = ~np.isnan(means)
            ax.plot(np.array(NOISE_RATES)[valid], means[valid],
                    label=m_name, color=color, marker='o', linewidth=1.8)
            ax.fill_between(np.array(NOISE_RATES)[valid],
                            (means - stds)[valid], (means + stds)[valid],
                            color=color, alpha=0.15)
        ax.set_title(nl); ax.set_xlabel('Noise Rate')
        ax.set_xticks(NOISE_RATES); ax.set_xlim(-0.02, 0.52)
        ax.set_ylim(0.0, 1.05); ax.grid(True, linestyle='--', alpha=0.4)

    axes[0].set_ylabel('Validation Accuracy')
    axes[-1].legend(loc='lower left', fontsize=8)
    plt.tight_layout()
    plt.savefig(MYLIBS / 'results' / 'fig1_accuracy_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

### Figure 2 — GMM Loss Histogram

In [ ]:
from sklearn.mixture import GaussianMixture
from scipy.stats import norm as scipy_norm

AUX_FILE = MYLIBS / 'results' / 'synthetic_aux.json'

if not AUX_FILE.exists():
    print("Run Phase 4A first.")
else:
    with open(AUX_FILE) as f:
        aux = json.load(f)

    HIST_RATES = [0.1, 0.2, 0.3, 0.4, 0.5]
    fig, axes  = plt.subplots(1, len(HIST_RATES), figsize=(4 * len(HIST_RATES), 4))
    fig.suptitle('Figure 2 — Per-Sample Loss Distribution (Uniform Noise, BaselineCE)', fontsize=11)

    for ax, nr in zip(axes, HIST_RATES):
        key    = f"uniform_{nr}_BaselineCE"
        losses = aux.get('losses', {}).get(key)
        if losses is None:
            ax.set_title(f'nr={nr}\n(no data)'); continue

        losses = np.array(losses)
        noisy_tr = apply_noise(train_features, train_labels, 'uniform', nr, seed=0)
        is_noisy = noisy_tr != train_labels

        bins = np.linspace(0, min(losses.max(), 8.0), 60)
        ax.hist(losses[~is_noisy], bins=bins, alpha=0.6, color='steelblue',
                label='Clean', density=True)
        ax.hist(losses[is_noisy],  bins=bins, alpha=0.6, color='salmon',
                label='Noisy', density=True)

        gmm = GaussianMixture(n_components=2, random_state=42).fit(losses.reshape(-1, 1))
        x   = np.linspace(0, min(losses.max(), 8.0), 300)
        for k in range(2):
            mu  = gmm.means_[k, 0]
            sig = np.sqrt(gmm.covariances_[k, 0, 0])
            ax.plot(x, gmm.weights_[k] * scipy_norm.pdf(x, mu, sig),
                    'k--', linewidth=1.5, alpha=0.8)

        ax.set_title(f'Noise Rate = {int(nr*100)}%')
        ax.set_xlabel('CE Loss')
        if ax is axes[0]: ax.set_ylabel('Density')
        ax.legend(fontsize=7)

    plt.tight_layout()
    plt.savefig(MYLIBS / 'results' / 'fig2_loss_histogram.png', dpi=150, bbox_inches='tight')
    plt.show()

### Figure 3 — Transition Matrix Heatmap

In [ ]:
if not AUX_FILE.exists():
    print("Run Phase 4A first.")
else:
    true_mat = aux.get('cl_true_matrix')
    est_mat  = aux.get('cl_count_matrix')

    if true_mat is None or est_mat is None:
        print("Transition matrix data not found — re-run the sweep.")
    else:
        def plot_mat(ax, mat, classes, title, vmax):
            mat = np.array(mat, dtype=float)
            im  = ax.imshow(mat, cmap='Blues', vmin=0, vmax=vmax)
            ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
            ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
            ax.set_yticklabels(classes, fontsize=8)
            ax.set_xlabel('Assigned Label'); ax.set_ylabel('True Label')
            ax.set_title(title)
            for i in range(len(classes)):
                for j in range(len(classes)):
                    c = 'white' if mat[i,j] > mat.max()*0.6 else 'black'
                    ax.text(j, i, str(int(mat[i,j])), ha='center', va='center', fontsize=7, color=c)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        vmax = max(np.array(true_mat).max(), np.array(est_mat).max())
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle('Figure 3 — Noise Transition Matrix at 30% Uniform Noise', fontsize=12)
        plot_mat(ax1, true_mat, CLASSES, 'Ground Truth',                vmax)
        plot_mat(ax2, est_mat,  CLASSES, "Confident Learning's Estimate", vmax)
        plt.tight_layout()
        plt.savefig(MYLIBS / 'results' / 'fig3_transition_matrix.png', dpi=150, bbox_inches='tight')
        plt.show()

### Figure 4 — Flagged Sample Grid

In [ ]:
from PIL import Image as PILImage

if not AUX_FILE.exists():
    print("Run Phase 4A first.")
else:
    flagged_global = aux.get('cl_flagged')
    if not flagged_global:
        print("No flagged samples saved — re-run sweep.")
    else:
        flagged = np.array(flagged_global, dtype=np.int64)
        noisy_30 = apply_noise(train_features, train_labels, 'uniform', 0.3, seed=0)

        GRID_ROWS, GRID_COLS = 4, 6
        show_idx = flagged[:GRID_ROWS * GRID_COLS]
        n_show   = len(show_idx)
        rows     = (n_show + GRID_COLS - 1) // GRID_COLS

        fig, axes = plt.subplots(rows, GRID_COLS, figsize=(GRID_COLS * 2.2, rows * 2.6))
        fig.suptitle('Figure 4 — Samples Flagged by Confident Learning\n(30% Uniform Noise)',
                     fontsize=11)
        axes_flat = np.array(axes).ravel()

        for i, train_pos in enumerate(show_idx):
            global_idx = train_idx[train_pos]
            img_path   = str(image_paths[global_idx])
            given_lbl  = CLASSES[noisy_30[train_pos]]
            true_lbl   = CLASSES[train_labels[train_pos]]
            ax = axes_flat[i]
            try:
                ax.imshow(PILImage.open(img_path).convert('RGB'))
            except Exception:
                ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes, ha='center', va='center')
            color = 'red' if given_lbl != true_lbl else 'green'
            ax.set_title(f'Given: {given_lbl}\nTrue: {true_lbl}', fontsize=7, color=color)
            ax.axis('off')

        for j in range(n_show, len(axes_flat)):
            axes_flat[j].axis('off')

        plt.tight_layout()
        plt.savefig(MYLIBS / 'results' / 'fig4_flagged_samples.png', dpi=120, bbox_inches='tight')
        plt.show()

### Figure 5 — Noise Rate Calibration

In [ ]:
if not AUX_FILE.exists():
    print("Run Phase 4A first.")
else:
    cl_est    = aux.get('cl_noise_est', {})
    nr_stored = aux.get('noise_rates', [0.0, 0.1, 0.2, 0.3, 0.4, 0.5])
    nt_list   = aux.get('noise_types', ['uniform', 'asymmetric', 'instance'])
    nt_labels = {'uniform': 'Uniform', 'asymmetric': 'Asymmetric', 'instance': 'Instance-Dep.'}
    colors_cal = cm.tab10(np.linspace(0, 0.6, len(nt_list)))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_title('Figure 5 — Confident Learning Noise Rate Calibration', fontsize=12)

    for nt, color in zip(nt_list, colors_cal):
        true_r, est_r = [], []
        for nr in nr_stored:
            key = f"{nt}_{nr}"
            if key in cl_est:
                true_r.append(nr); est_r.append(cl_est[key])
        if true_r:
            ax.scatter(true_r, est_r, color=color, s=80, zorder=3,
                       label=nt_labels.get(nt, nt))
            ax.plot(true_r, est_r, color=color, linewidth=1, alpha=0.5)

    lim = max(max(nr_stored) + 0.05, 0.6)
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1.2, label='Ideal (y=x)', zorder=2)
    ax.set_xlim(-0.02, lim); ax.set_ylim(-0.02, lim)
    ax.set_xlabel('True Noise Rate'); ax.set_ylabel('CL Estimated Noise Rate')
    ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig(MYLIBS / 'results' / 'fig5_calibration.png', dpi=150, bbox_inches='tight')
    plt.show()

### Figure 6 — Learning Curves

In [ ]:
if not AUX_FILE.exists():
    print("Run Phase 4A first.")
else:
    curves = aux.get('epoch_curves', {})
    if not curves:
        print("No epoch curves saved — re-run sweep.")
    else:
        METHOD_ORDER = ['BaselineCE', 'LabelSmoothing', 'SCE', 'GCE',
                        'GMMReweight', 'ConfidentLearning']
        colors_lc = cm.tab10(np.linspace(0, 0.8, len(METHOD_ORDER)))

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.set_title('Figure 6 — Learning Curves at 40% Uniform Noise (Seed 0)', fontsize=12)

        for m_name, color in zip(METHOD_ORDER, colors_lc):
            if m_name in curves:
                accs = curves[m_name]
                ax.plot(range(1, len(accs) + 1), accs, label=m_name,
                        color=color, linewidth=1.8)

        ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy')
        ax.set_ylim(0.0, 1.05); ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(fontsize=9, loc='lower right')
        plt.tight_layout()
        plt.savefig(MYLIBS / 'results' / 'fig6_learning_curves.png', dpi=150, bbox_inches='tight')
        plt.show()

### Figure 7 — Real Data Bar Chart

In [ ]:
REAL_FILE = MYLIBS / 'results' / 'real_results.json'

if not REAL_FILE.exists():
    print("Run Phase 4B first.")
else:
    with open(REAL_FILE) as f:
        real_data = json.load(f)

    results    = real_data['results']
    noise_info = real_data['noise_rates']
    label_sets = real_data['label_sets']
    METHOD_ORDER = ['BaselineCE', 'LabelSmoothing', 'SCE', 'GCE',
                    'GMMReweight', 'ConfidentLearning']
    LS_DISPLAY = {'clean': 'Clean', 'human': 'Human\nAnnotations',
                  'uniform_30': 'Uniform\n30%', 'asymmetric_30': 'Asymmetric\n30%'}

    n_groups  = len(label_sets)
    n_methods = len(METHOD_ORDER)
    bar_width = 0.8 / n_methods
    colors_bar = cm.tab10(np.linspace(0, 0.8, n_methods))

    fig, ax = plt.subplots(figsize=(max(10, n_groups * 3), 5))
    ax.set_title('Figure 7 — Validation Accuracy on Real Human Annotation Noise', fontsize=12)

    x_base = np.arange(n_groups)
    for m_i, (m_name, color) in enumerate(zip(METHOD_ORDER, colors_bar)):
        accs   = [results.get(ls, {}).get(m_name, 0.0) for ls in label_sets]
        offset = (m_i - n_methods / 2 + 0.5) * bar_width
        ax.bar(x_base + offset, accs, width=bar_width * 0.9, color=color, label=m_name)

    xlabels = []
    for ls in label_sets:
        base = LS_DISPLAY.get(ls, ls)
        nr   = noise_info.get(ls)
        xlabels.append(f"{base}\n(nr={nr:.2f})" if nr is not None else base)

    ax.set_xticks(x_base); ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_ylabel('Validation Accuracy'); ax.set_ylim(0.0, 1.05)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    plt.tight_layout()
    plt.savefig(MYLIBS / 'results' / 'fig7_real_data_bar.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Summary

All figures are saved as PNGs in `mylibs/results/`:

| File | Figure |
|---|---|
| `fig1_accuracy_curves.png` | Accuracy vs noise rate (3 noise types) |
| `fig2_loss_histogram.png` | GMM loss histograms (5 noise rates) |
| `fig3_transition_matrix.png` | CL estimated vs true transition matrix |
| `fig4_flagged_samples.png` | Image grid of samples flagged by CL |
| `fig5_calibration.png` | CL noise rate calibration scatter |
| `fig6_learning_curves.png` | Val accuracy per epoch at 40% uniform noise |
| `fig7_real_data_bar.png` | Method comparison on real human annotations |

Raw results are in `mylibs/results/synthetic_results.json`, `mylibs/results/real_results.json`, and `mylibs/results/synthetic_aux.json`.